### class method

define a method that
operates on the class and not on instances. classmethod changes the way the method
is called, so it receives the class itself as the first argument, instead of an instance. Its
most common use is for alternative constructors,

@staticmethod → rarely useful

If it doesn’t use the class, why not just make it a normal function outside the class?

In [2]:
class SomeClass:
    @classmethod
    def methodc(*args):
        return args
    @staticmethod
    def methods(*args):
        return args

SomeClass.methodc(), SomeClass.methods()

((__main__.SomeClass,), ())

In [3]:
# === Python Formatting Basics ===

# 1) format() and f-strings use __format__ internally
x = 2 / 3
print(format(x, ".2f"))     # 0.67
print(f"{x:.2f}")           # 0.67

# format_spec = ".2f" (2 decimal places)


# 2) Built-in types support formatting
print(format(42, "b"))      # binary → 101010
print(format(2/3, ".1%"))   # percentage → 66.7%


# === Custom Class with __format__ ===

class Vector2d:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __iter__(self):
        return (i for i in (self.x, self.y))

    def __abs__(self):
        return (self.x**2 + self.y**2) ** 0.5

    def angle(self):
        import math
        return math.atan2(self.y, self.x)

    def __format__(self, fmt_spec=""):
        if fmt_spec.endswith("p"):          # custom: polar coordinates
            fmt_spec = fmt_spec[:-1]
            coords = (abs(self), self.angle())
            outer = "<{}, {}>"
        else:                              # default: (x, y)
            coords = self
            outer = "({}, {})"

        components = (format(c, fmt_spec) for c in coords)
        return outer.format(*components)


# === Usage ===
v = Vector2d(3, 4)

print(format(v))            # (3.0, 4.0)
print(format(v, ".2f"))     # (3.00, 4.00)
print(format(v, ".2fp"))    # <5.00, 0.93>  (polar form)

0.67
0.67
101010
66.7%
(3.0, 4.0)
(3.00, 4.00)
<5.00, 0.93>


In [4]:
class Vector2d:
    def __init__(self, x, y):
        self.__x = x
        self.__y = y

    @property
    def x(self):
        return self.__x

    @property
    def y(self):
        return self.__y

    def __hash__(self):
        return hash((self.x, self.y))


v1 = Vector2d(3, 4)
v2 = Vector2d(3, 4)

print(hash(v1))
print(v1.x)

# Try this:
# v1.x = 10   # should fail

# Now this works:
s = {v1, v2}
print(s)

1079245023883434373
3
{<__main__.Vector2d object at 0x000001DE07AD75F0>, <__main__.Vector2d object at 0x000001DE07AD7410>}


__hash__ → “how do I turn this into a number?”
@property → “you can read, but not change”
__x → “keep it private”

__match_args__ tells Python how to interpret positional values in pattern matching.

In [5]:
class User:
    __match_args__ = ("role", "name")

    def __init__(self, role, name):
        self.role = role
        self.name = name

    def __repr__(self):
        return f"User({self.role}, {self.name})"

def check_user(u):
    match u:
        case User("admin", name):
            print("Admin user:", name)

        case User("guest", _):
            print("Guest user")

        case User(role, name):
            print(f"Normal user: {name} ({role})")

u1 = User("admin", "Ali")
u2 = User("guest", "Sara")
u3 = User("member", "Omar")

check_user(u1)
check_user(u2)
check_user(u3)

Admin user: Ali
Guest user
Normal user: Omar (member)


### Name mangling.

Consider this scenario: someone wrote a class named Dog that uses a mood instance
attribute internally, without exposing it. You need to subclass Dog as Beagle. If you
create your own mood instance attribute without being aware of the name clash, you
will clobber the mood attribute used by the methods inherited from Dog. This would
be a pain to debug.
To prevent this, if you name an instance attribute in the form __mood (two leading
underscores and zero or at most one trailing underscore), Python stores the name in
the instance __dict__ prefixed with a leading underscore and the class name, so in
the Dog class, __mood becomes _Dog__mood, and in Beagle it’s _Beagle__mood. This
language feature goes by the lovely name of name mangling.

In [ ]:
class Dog:
    def __init__(self):
        self.__mood = "happy"

    def show_mood(self):
        print(self.__mood)

class Beagle(Dog):
    def __init__(self):
        super().__init__()
        self.__mood = "angry"

b = Beagle()
b.show_mood()

d = Dog()
d._Dog__mood
#d.__mood       -> error


happy


'happy'

__slots__ saves memory instead of dict

In [13]:
class Pixel:
    __slots__ = ("x", "y")

p = Pixel()
p.x = 11
p.y = 22
#p.color = 'green'   -> error

In [ ]:
class PColor(Pixel):
    pass

pp = PColor()
pp.color = 'red' #works becuase __slots__ not inherted

Instead of modifying the class directly, you can create a subclass: if self.attribute = something.  subclass then override